Data source
-----------
HRSA Data Warehouse → Health Center Service Delivery and Look-Alike

Sites URL: https://data.hrsa.gov/data/download

Download: CSV (~10.5 MB), updated daily
Save as: data/raw/hrsa_health_center_service_delivery_sites.csv

The dataset lists individual federally-funded health center sites with
state abbreviation, congressional district, and service type.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path(__file__).resolve().parents[2] / "shared"))

import pandas as pd
from config import DATA_RAW, DATA_INTERIM, STATE_ABBR_TO_FIPS


def find_hrsa_file() -> Path:
    patterns = [
        "*hrsa*service*delivery*",
        "*health*center*service*delivery*",
        "*hrsa_sites*",
        "*hrsa*",
    ]
    for pat in patterns:
        matches = list(DATA_RAW.glob(pat))
        if matches:
            return matches[0]
    raise FileNotFoundError(
        "No HRSA file found in data/raw/. Download from:\n"
        "https://data.hrsa.gov/data/download\n"
        "Search: 'Health Center Service Delivery' → CSV"
    )


def find_state_column(df: pd.DataFrame) -> str:


Flexible column detection for varying HRSA export formats.


In [ ]:
    """Flexible column detection for varying HRSA export formats."""
        "site state abbreviation", "state abbreviation", "site state",
        "state abbr", "state", "st",
    ]
    for c in df.columns:
        if c.lower().strip() in candidates:
            return c
    # Fallback: any column with 'state' and short values
    for c in df.columns:
        if "state" in c.lower():
            sample = df[c].dropna().astype(str).str.strip()
            if sample.str.len().max() <= 3:
                return c
    raise ValueError(f"Cannot identify state column. Available columns: {list(df.columns)}")


def main():
    path = find_hrsa_file()
    print(f"Loading HRSA from: {path.name}")

    df = pd.read_csv(path, low_memory=False, encoding="latin1")
    print(f"  Loaded {len(df):,} rows, {len(df.columns)} columns")

    state_col = find_state_column(df)
    print(f"  Using state column: '{state_col}'")

    df["state_fips"] = (
        df[state_col].astype(str).str.strip().str.upper()
        .map(STATE_ABBR_TO_FIPS)
    )

    dropped = df["state_fips"].isna().sum()
    if dropped > 0:
        unmapped = df[df["state_fips"].isna()][state_col].value_counts().head(5)
        print(f"  Warning: {dropped} rows have unmapped state codes:\n{unmapped}")

    state_df = (
        df[df["state_fips"].notna()]
        .groupby("state_fips")
        .size()
        .reset_index(name="fqhc_site_count")
    )

    out = DATA_INTERIM / "hrsa_state_visibility.csv"
    state_df.to_csv(out, index=False)
    print(f"  Saved → {out}  ({len(state_df)} states)")
    print(state_df.sort_values("fqhc_site_count", ascending=False).head(8).to_string(index=False))


if __name__ == "__main__":
    main()
